# 05 — SQL Fundamentals

### Why this exists

SQL is a language for asking questions about sets of rows. The important engineering habit is to reason about grain, NULLs and aggregation before trusting the answer.

### Learning objectives

- filter and aggregate correctly
- reason about NULL
- state the grain of a result

### Prerequisite

03 — CSV vs JSON vs Parquet

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — SQL Fundamentals](../assets/sketch-notes/05-sql-fundamentals.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S01 — Ticket DATA-1042: Revenue dashboard mismatch** → [`../work_simulations/01_acme_commerce_revenue_mismatch/README.md`](../work_simulations/01_acme_commerce_revenue_mismatch/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

Write down one thing you expect to observe and why. Do not use the output as your prediction.


## Production bridge

Ask: what happens when the data is 100× larger, arrives late, changes shape, or is queried by several teams? State the assumption that makes your experiment useful.


## Independent challenge

Add a query that answers a new business question and explain its grain.

### Evidence to keep

- your prediction
- one measurement
- the failure or edge case you tested
- one production implication


# 05 — SQL Fundamentals

## The minimum theory

SQL is **declarative**: you describe the result you want and the database engine decides how to execute it.

Core ideas:
- `WHERE` filters rows.
- `GROUP BY` defines groups.
- aggregate functions such as `SUM` and `COUNT` summarize groups.
- `ORDER BY` controls result ordering.

This separation of **what** from **how** is why database engines can later optimize queries.

We will use DuckDB because it is lightweight and works well in Colab.


In [ ]:
import sys,subprocess,importlib.util
if importlib.util.find_spec("duckdb") is None: subprocess.check_call([sys.executable,"-m","pip","install","-q","duckdb>=1,<2"])
import duckdb,pandas as pd
orders=pd.DataFrame({"order_id":range(1,11),"customer_id":[101,102,101,103,104,102,105,101,104,103],"country":["IN","US","IN","DE","IN","US","GB","IN","IN","DE"],"status":["paid","paid","cancelled","paid","paid","paid","paid","paid","cancelled","paid"],"amount":[120,80,50,200,150,90,300,70,110,180]})
con=duckdb.connect(); con.register("orders",orders)


## Filter and aggregate

Start with individual records, then ask a business question at summary level: “What is paid revenue by country?”


In [ ]:
print(con.sql("SELECT order_id,country,amount FROM orders WHERE status='paid' AND amount>=100 ORDER BY amount DESC").df())
print(con.sql("SELECT country,COUNT(*) AS paid_orders,SUM(amount) AS revenue FROM orders WHERE status='paid' GROUP BY country ORDER BY revenue DESC").df())


## `NULL` is not zero

`NULL` represents missing/unknown information. `COUNT(*)` counts rows, while `COUNT(column)` counts non-null values.

Try changing a value to `NULL` and compare the results.


In [ ]:
x=orders.copy(); x.loc[0,"amount"]=None; con.register("orders_null",x)
con.sql("SELECT COUNT(*) AS rows,COUNT(amount) AS non_null_amounts,SUM(amount) AS total_amount FROM orders_null").df()


## Break it

Try `COUNT(amount)`, a `NULL` country, and `amount = NULL` versus `amount IS NULL`.

The goal is to learn semantics by observation.

**Next:** joins and aggregations.
